# Data Quality Assessment

## Business Question

Is the available data complete, accurate, and reliable enough to support business decisions?

## Objectives

- Identify missing values
- Detect duplicate records
- Validate data types
- Investigate outliers
- Assess data reliability before analysis

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

if (Path.cwd() / "data" / "raw").exists():
    DATA_PATH = Path.cwd() / "data" / "raw"
elif (Path.cwd().parent / "data" / "raw").exists():
    DATA_PATH = Path.cwd().parent / "data" / "raw"
else:
    raise FileNotFoundError("Could not find the data/raw folder.")

customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    DATA_PATH / "product_category_name_translation.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [2]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

for name, df in datasets.items():
    print(f'\n{name.upper()}')
    print('-' * 50)
    print(f'Shape: {df.shape}')


CUSTOMERS
--------------------------------------------------
Shape: (99441, 5)

GEOLOCATION
--------------------------------------------------
Shape: (1000163, 5)

ORDER_ITEMS
--------------------------------------------------
Shape: (112650, 7)

PAYMENTS
--------------------------------------------------
Shape: (103886, 5)

REVIEWS
--------------------------------------------------
Shape: (99224, 7)

ORDERS
--------------------------------------------------
Shape: (99441, 8)

PRODUCTS
--------------------------------------------------
Shape: (32951, 9)

SELLERS
--------------------------------------------------
Shape: (3095, 4)

CATEGORY_TRANSLATION
--------------------------------------------------
Shape: (71, 2)


### Dataset Validation
Did all datasets load correctly and are they structurally consistent?

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

In [4]:
from IPython.display import display

In [5]:
data_quality_overview = []

for name, df in datasets.items():
    data_quality_overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "missing_pct": round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2)
    })

data_quality_overview = pd.DataFrame(data_quality_overview)
data_quality_overview.sort_values("missing_values", ascending=False)

,dataset,rows,columns,duplicate_rows,missing_values,missing_pct
4,reviews,99224,7,0,145903,21.01
5,orders,99441,8,0,4908,0.62
6,products,32951,9,0,2448,0.83
0,customers,99441,5,0,0,0.00
1,geolocation,1000163,5,261831,0,0.00
2,order_items,112650,7,0,0,0.00
3,payments,103886,5,0,0,0.00
7,sellers,3095,4,0,0,0.00
8,category_translation,71,2,0,0,0.00


In [6]:
missing_summary = []

for dataset_name, df in datasets.items():
    missing_counts = df.isnull().sum()
    missing_percent = (df.isnull().mean() * 100).round(2)

    for column in df.columns:
        if missing_counts[column] > 0:
            missing_summary.append({
                "dataset": dataset_name,
                "column": column,
                "missing_count": missing_counts[column],
                "missing_percent": missing_percent[column]
            })

missing_summary_df = pd.DataFrame(missing_summary)
missing_summary_df.sort_values(
    by=["missing_count", "missing_percent"],
    ascending=False
)

,dataset,column,missing_count,missing_percent
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
4,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
2,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


### Missing Values Finding

Most datasets are complete. Missing values are concentrated in review comment fields, order delivery timestamps, and product attribute fields.

This is expected in several cases:
- not all customers leave written review comments
- undelivered or cancelled orders may not have delivery timestamps
- some products have incomplete catalog attributes

These fields should be handled carefully before customer satisfaction, delivery performance, or product-level analysis.

In [7]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [8]:
orders[orders['order_delivered_customer_date'].isnull()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaN,NaN,2018-08-21 00:00:00
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaN,NaN,2017-10-03 00:00:00
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00
...,...,...,...,...,...,...,...,...
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00
99313,e9e64a17afa9653aacf2616d94c005b8,b4cd0522e632e481f8eaf766a2646e86,processing,2018-01-05 23:07:24,2018-01-09 07:18:05,NaN,NaN,2018-02-06 00:00:00
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaN,NaN,NaN,2018-09-27 00:00:00
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaN,NaN,2017-09-15 00:00:00


In [9]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [10]:
missing_delivery_orders = orders[orders['order_delivered_customer_date'].isnull()]

In [11]:
missing_delivery_orders['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [12]:
missing_delivery_orders[missing_delivery_orders['order_status'] == 'delivered']
delivered_missing_orders = missing_delivery_orders[missing_delivery_orders['order_status'] == 'delivered']

In [13]:
delivered_missing_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


### Finding

The Orders dataset contains 2,965 missing customer delivery timestamps. A detailed investigation showed that only **8** of these orders are marked as delivered.

Manual inspection of purchase, approval, carrier, and estimated delivery timestamps indicates that these records follow a normal order lifecycle. This suggests that the missing customer delivery timestamp is most likely a data recording issue rather than evidence of failed deliveries.

To validate this assumption, the corresponding customer reviews were examined.

In [14]:
delivered_missing_reviews = pd.merge(
    delivered_missing_orders,
    reviews,
    on='order_id',
    how='left'
)

In [15]:
delivered_missing_reviews

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00,f48c6c944a5d52dcca8ac5c4ec417cf2,5,NaN,Chegou rápido tudo ok,2017-12-19 00:00:00,2017-12-19 04:15:39
1,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00,c0dd6bec0375c376f044af102118526f,5,Entrega super rápida.,"Produto novo, muito bom.",2018-06-29 00:00:00,2018-06-29 16:26:37
2,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00,25e11638a3d01a87e8e62338a39eee28,5,NaN,NaN,2018-07-11 00:00:00,2018-07-11 19:27:46
3,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00,bb311d9562ecbefc8e4be756d8999892,5,NaN,NaN,2018-07-07 00:00:00,2018-07-10 11:38:13
4,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00,ee2d30652e2f7fc00861074f795f5bf0,5,Excelente!,O produto chegou muito antes do prazo previsto...,2018-07-06 00:00:00,2018-07-07 18:48:09
5,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00,4e755f114e50d33b9ac6a56e0d7d3ea9,5,NaN,NaN,2017-06-25 00:00:00,2017-06-27 01:49:04
6,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00,0d4c56af896dd6eb9de8edbaa1902d22,1,Péssimo,Comprei um produto de uma marca e recebi outro...,2018-06-16 00:00:00,2018-06-16 13:55:00
7,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00,d055795a562efffefe47ef81e5435322,5,Muito bom,Adorei,2018-07-06 00:00:00,2018-07-06 20:30:17


#### Translated comments from Brazilian into English:

| Review Score | Original                                                         | English Translation                                                           | Interpretation                                                                                                                                      |
| ------------ | ---------------------------------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| ⭐⭐⭐⭐⭐        | **Chegou rápido tudo ok**                                        | "Arrived quickly, everything is OK."                                          | Product was definitely received.                                                                                                                    |
| ⭐⭐⭐⭐⭐        | **Entrega super rápida. Produto novo, muito bom.**               | "Super fast delivery. Brand new product, very good."                          | Confirms successful delivery.                                                                                                                       |
| ⭐⭐⭐⭐⭐        | *(No comment)*                                                   | No written review.                                                            | Neutral. Still gave 5 stars.                                                                                                                        |
| ⭐⭐⭐⭐⭐        | *(No comment)*                                                   | No written review.                                                            | Neutral. Still gave 5 stars.                                                                                                                        |
| ⭐⭐⭐⭐⭐        | **Excelente! O produto chegou muito antes do prazo previsto...** | "Excellent! The product arrived well before the estimated delivery date..."   | Strong evidence the delivery occurred.                                                                                                              |
| ⭐⭐⭐⭐⭐        | *(No comment)*                                                   | No written review.                                                            | Neutral. Still gave 5 stars.                                                                                                                        |
| ⭐☆☆☆☆        | **Péssimo. Comprei um produto de uma marca e recebi outro...**   | "Terrible. I bought a product from one brand and received a different one..." | **Important:** The customer complains about receiving the **wrong product**, not about non-delivery. This still confirms the package was delivered. |
| ⭐⭐⭐⭐⭐        | **Muito bom** / **Adorei**                                       | "Very good." / "I loved it."                                                  | Confirms successful delivery.                                                                                                                       |


### Business Conclusion

Evidence from the Reviews dataset confirms that these orders were successfully delivered despite the missing delivery timestamp.

Customer comments consistently describe received products rather than missing deliveries. This indicates that the missing `order_delivered_customer_date` field is primarily a data quality issue and should not be interpreted as operational delivery failure.

These records can remain in the dataset but should be excluded from analyses requiring precise delivery duration calculations.

#### Confidence Level
**High**

Reason
- Timeline data is internally consistent.
- 7 of 8 orders contain carrier delivery timestamps.
- Customer reviews explicitly confirm successful deliveries.
- The only negative review concerns receiving the wrong product, not non-delivery.

## Duplicate Record Assessment

**Business Question**

Are there unexpected duplicate records that could affect analysis?

**Objectives**

- Identify duplicate rows
- Distinguish expected duplicates from data quality issues
- Document acceptable one-to-many relationships

In [16]:
duplicate_summary = []

for dataset_name, df in datasets.items():
    duplicate_summary.append({
        "dataset": dataset_name,
        "rows": len(df),
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_percent": round(df.duplicated().mean() * 100, 2)
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
duplicate_summary_df.sort_values("duplicate_rows", ascending=False)

,dataset,rows,duplicate_rows,duplicate_percent
1,geolocation,1000163,261831,26.18
0,customers,99441,0,0.00
2,order_items,112650,0,0.00
3,payments,103886,0,0.00
4,reviews,99224,0,0.00
5,orders,99441,0,0.00
6,products,32951,0,0.00
7,sellers,3095,0,0.00
8,category_translation,71,0,0.00


### Business Finding

No unexpected duplicate records were identified within the transactional datasets.

Duplicate rows were only observed in the geolocation dataset, where repeated combinations of postal code, city, state, latitude, and longitude are expected because this table functions as a geographic reference rather than a transactional table.

No cleaning action is required before proceeding with further analysis.

In [17]:
dtype_summary = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        dtype_summary.append({
            "dataset": dataset_name,
            "column": column,
            "dtype": str(df[column].dtype),
            "non_null_count": df[column].notnull().sum(),
            "missing_count": df[column].isnull().sum()
        })

dtype_summary_df = pd.DataFrame(dtype_summary)
dtype_summary_df

,dataset,column,dtype,non_null_count,missing_count
0,customers,customer_id,object,99441,0
1,customers,customer_unique_id,object,99441,0
2,customers,customer_zip_code_prefix,int64,99441,0
3,customers,customer_city,object,99441,0
4,customers,customer_state,object,99441,0
5,geolocation,geolocation_zip_code_prefix,int64,1000163,0
6,geolocation,geolocation_lat,float64,1000163,0
7,geolocation,geolocation_lng,float64,1000163,0
8,geolocation,geolocation_city,object,1000163,0
9,geolocation,geolocation_state,object,1000163,0


#### Finding
All timestamp columns are currently stored as object rather than datetime. They are readable but unsuitable for time-based analysis without conversion.

**Recommendation**

- Converting timestamp fields to datetime format will enable accurate delivery time calculations, trend analysis, seasonality analysis, and time-based KPI reporting during the Data Preparation stage.
- Convert all timestamp columns to datetime64[ns] during the Data Preparation stage before performing delivery time, seasonality, or trend analysis.

## Logical Consistency Assessment

**Business Question**

Does the data follow valid business rules and chronological order?

**Objectives**

- Validate timestamp sequence
- Detect impossible values
- Verify business logic
- Identify inconsistencies requiring correction

In [18]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

In [19]:
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at']
)
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date']
)
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date']
)
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date']
)

In [20]:
approval_before_purchase = orders[
    orders['order_approved_at'] < orders['order_purchase_timestamp']
    ]

approval_before_purchase

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [21]:
carrier_before_approval = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_approved_at"]
].copy()

carrier_before_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15


In [22]:
customer_before_carrier = orders[
    orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']
]

customer_before_carrier

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,6cd7106899e59a1fbd0622d5f1efedf4,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,9859dd92e872dbaa60ca3cd5f0d7ad07,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,01c843a2c0600def0b7693dba47af460,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,3947a361301f2ff0f3223159a0f2701c,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,63be4feff10a0b1d85f2cfbf10df9754,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


In [23]:
estimated_before_purchase = orders[
    orders['order_estimated_delivery_date'] < orders['order_purchase_timestamp']
]

estimated_before_purchase

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [24]:
print(f"Orders approved before purchase: {len(approval_before_purchase)}")
print(f"Carrier before approval: {len(carrier_before_approval)}")
print(f"Customer before carrier: {len(customer_before_carrier)}")
print(f"Estimated delivery before purchase: {len(estimated_before_purchase)}")

Orders approved before purchase: 0
Carrier before approval: 1359
Customer before carrier: 23
Estimated delivery before purchase: 0


### Initial Findings

The chronological validation produced the following results:

- No orders were approved before purchase.
- No estimated delivery dates occur before purchase.
- 1,359 orders show a carrier shipment timestamp occurring before order approval.
- 23 orders show customer delivery occurring before carrier delivery.

The first two checks confirm that the core purchasing workflow is internally consistent. The remaining anomalies require further investigation.

In [25]:
carrier_before_approval['approval_delay'] = (
    carrier_before_approval['order_approved_at']
    - carrier_before_approval['order_delivered_carrier_date']
)

In [26]:
carrier_before_approval = orders[
    orders['order_delivered_carrier_date'] < orders['order_approved_at']
].copy()

In [27]:
carrier_before_approval['approval_delay'] = (
    carrier_before_approval['order_approved_at']
    - carrier_before_approval['order_delivered_carrier_date']
)

carrier_before_approval.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04,1 days 08:37:02
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15,0 days 23:06:08
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31,2 days 10:34:53
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31,0 days 00:07:53
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06,3 days 09:28:09


In [28]:
carrier_before_approval['approval_delay'].describe()

count                         1359
mean     1 days 00:45:07.153053715
std      4 days 19:18:28.055754668
min                0 days 00:00:21
25%         0 days 01:24:55.500000
50%                0 days 17:10:04
75%                1 days 01:57:24
max              171 days 05:15:22
Name: approval_delay, dtype: object

In [29]:
carrier_before_approval['approval_delay'].sort_values()

3486      0 days 00:00:21
72849     0 days 00:01:11
38741     0 days 00:01:12
28238     0 days 00:01:13
38841     0 days 00:01:21
               ...       
41592     9 days 02:45:45
98710     9 days 03:55:56
46163     9 days 08:11:25
14562     9 days 08:55:48
25883   171 days 05:15:22
Name: approval_delay, Length: 1359, dtype: timedelta64[ns]

In [30]:
carrier_before_approval['approval_delay'].value_counts().head(20)

approval_delay
1 days 05:11:53    2
0 days 01:19:35    2
0 days 01:36:01    2
0 days 01:43:03    2
0 days 00:53:14    2
0 days 02:26:03    2
0 days 00:30:12    2
0 days 02:08:53    2
0 days 02:03:13    2
1 days 02:17:47    2
0 days 00:32:32    2
0 days 01:28:37    2
0 days 19:21:46    2
0 days 23:06:08    2
0 days 03:51:50    2
0 days 01:40:08    2
0 days 01:09:45    2
0 days 02:01:29    2
0 days 00:57:03    2
0 days 00:48:26    2
Name: count, dtype: int64

### Finding 1

#### Interpretation

Most approval delays are shorter than one day.

These short delays are unlikely to represent operational problems and are more likely explained by:

- delayed system synchronization
- overnight batch processing
- timestamp recording after operational completion

### Finding 2

#### Investigation Result

A very small number of records show unusually long approval delays, with one extreme case reaching 171 days.

Because these observations are exceptionally rare compared with the overall dataset, they should be treated as isolated anomalies rather than representative operational behaviour.

In [31]:
carrier_before_approval[
    carrier_before_approval['approval_delay'].dt.days > 30
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay
25883,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,171 days 05:15:22


#### One extreme outlier (171-day delay) was identified. Due to its magnitude and the absence of similar cases, it is treated as an isolated anomaly rather than representative of the dataset.

In [32]:
carrier_before_approval.sort_values(['order_approved_at'], ascending = True)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay
25411,6b80bb20190715d71c43efff617bd659,2fedecfd993b8b3fa889d00eee230748,delivered,2017-02-19 01:15:03,2017-03-01 10:51:46,2017-02-22 16:05:29,2017-02-24 14:27:26,2017-03-17,6 days 18:46:17
94980,2d0d4075ded592212bcd5e5bc561b406,ad4fdd8ba1535077790107fc697fbe97,delivered,2017-04-26 12:22:32,2017-04-26 13:06:25,2017-04-26 13:01:25,2017-05-15 07:38:03,2017-05-25,0 days 00:05:00
4466,69a236fbbc4a603ebfa4468a3bdcb140,1bd6b1b425cc25c6e0c79f68d28fe2fb,delivered,2017-04-25 01:46:02,2017-04-27 10:32:00,2017-04-26 09:11:44,2017-05-03 13:39:47,2017-05-25,1 days 01:20:16
63934,a60d571514c73cfc7655d67c75ed82c7,ad5945bdec9120fbc7eab1a9746695a7,delivered,2017-04-25 09:50:01,2017-04-27 10:32:03,2017-04-27 07:18:06,2017-05-05 18:58:07,2017-05-26,0 days 03:13:57
71317,af2adc7e31b52bdfe068cf60426b54b2,afd3f4f110c36e0ec5b412f4d30af5fc,delivered,2017-04-24 09:14:06,2017-04-27 10:32:39,2017-04-26 18:47:58,2017-05-02 17:24:56,2017-05-11,0 days 15:44:41
...,...,...,...,...,...,...,...,...,...
64701,70f814478ea3019fff8677f0aca6a54b,adad0c56d848f4034c2cb989a6139c98,delivered,2018-08-23 15:58:16,2018-08-23 16:10:22,2018-08-23 15:31:00,2018-08-27 18:54:37,2018-08-28,0 days 00:39:22
47604,cf5c8d9f52807cb2d2f0a0ff54c478da,e898b5ef24833b9cb9e2d4f00b937595,delivered,2018-08-24 13:04:05,2018-08-24 13:24:27,2018-08-24 12:00:00,2018-08-30 19:11:50,2018-10-05,0 days 01:24:27
80017,421deffec215845f8632fb3543789f86,07d32200708d263fc5006988e4b6eff7,delivered,2018-08-24 14:05:59,2018-08-24 14:24:29,2018-08-24 13:46:00,2018-08-30 20:32:08,2018-09-11,0 days 00:38:29
4256,4e157a36ea9cf89bde6fff57a780b525,24970f1325070d89a1c42dd450b09a8a,delivered,2018-08-24 14:37:50,2018-08-24 14:50:14,2018-08-24 12:43:00,2018-08-30 20:47:26,2018-09-10,0 days 02:07:14


In [33]:
carrier_before_approval.sort_values(['order_estimated_delivery_date'], ascending = True)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay
25411,6b80bb20190715d71c43efff617bd659,2fedecfd993b8b3fa889d00eee230748,delivered,2017-02-19 01:15:03,2017-03-01 10:51:46,2017-02-22 16:05:29,2017-02-24 14:27:26,2017-03-17,6 days 18:46:17
71317,af2adc7e31b52bdfe068cf60426b54b2,afd3f4f110c36e0ec5b412f4d30af5fc,delivered,2017-04-24 09:14:06,2017-04-27 10:32:39,2017-04-26 18:47:58,2017-05-02 17:24:56,2017-05-11,0 days 15:44:41
41421,1bab77281b20f4719d0a9f87f2282fde,28d0ce13f2e87804e85a7a2a0282c711,delivered,2017-04-24 22:25:28,2017-04-27 10:33:13,2017-04-27 07:42:03,2017-05-08 16:07:05,2017-05-11,0 days 02:51:10
68420,104cf29a266be8f412ca30ec4cdab145,394c4558e06b0c9e8d6673e0d1c75d62,delivered,2017-04-25 22:50:48,2017-04-27 13:36:40,2017-04-26 09:47:10,2017-05-08 07:46:56,2017-05-12,1 days 03:49:30
66029,cc460ac4435835dfdda5526dfe84f01f,99a32bf8f0c54702217b584a4d220761,delivered,2017-04-25 10:10:12,2017-04-27 10:32:44,2017-04-26 09:16:59,2017-04-27 22:23:09,2017-05-12,1 days 01:15:45
...,...,...,...,...,...,...,...,...,...
37710,8c8005895f46f6c1bdf16697c70e645a,74245fe8288c7bdcbcd4644b2edf5beb,delivered,2018-08-17 15:57:10,2018-08-17 16:10:14,2018-08-17 14:56:00,2018-08-25 16:08:37,2018-09-19,0 days 01:14:14
96100,c51f25e565a1b4a03be6979f4661a585,38cac1776e7f4cca67c47a655eca20b0,delivered,2018-08-20 11:33:30,2018-08-20 15:15:36,2018-08-20 14:52:00,2018-08-23 20:22:38,2018-09-24,0 days 00:23:36
36866,48b4f1f96d5ae13b404fe7d041a8d393,77151da438a727054ce6430fd175793a,delivered,2018-08-09 13:41:59,2018-08-09 14:35:22,2018-08-09 13:57:00,2018-08-24 16:44:28,2018-09-26,0 days 00:38:22
59989,8b5512b2679b86761e6aadc28ed91762,ee2eee66d80a60f1761eedb7599a971d,delivered,2018-08-03 14:39:54,2018-08-03 14:50:14,2018-08-03 13:57:00,2018-08-22 18:03:43,2018-10-01,0 days 00:53:14


In [34]:
carrier_before_approval.sort_values(
    'approval_delay',
    ascending=False
).head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay
25883,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,171 days 05:15:22


In [35]:
result = carrier_before_approval.merge(
    order_items,
    on='order_id',
    how='left'
).merge(
    payments,
    on='order_id',
    how='left'
).merge(
    reviews,
    on='order_id',
    how='left'
)

In [36]:
result.sort_values(
    'approval_delay',
    ascending=False
).head(1)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
439,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07,171 days 05:15:22,1,9070fbe936bd54b8a72e0ffe4a6a2564,89de2d6f23e9746ff309705b23581faa,2018-07-20 18:50:22,40.0,14.58,1,credit_card,1,54.58,49eb9d9ffb7e7d2cfd2dcfb484473497,5.0,NaN,NaN,2018-07-24 00:00:00,2018-07-25 23:04:33


In [37]:
result.sort_values(
    'approval_delay',
    ascending=True
).head(20)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
63,73efca6402c1b6108195348448d76147,4280fa78d7c43abc5e98a8a3b44d9350,delivered,2018-08-04 14:49:43,2018-08-06 14:55:21,2018-08-06 14:55:00,2018-08-09 23:53:30,2018-08-16,0 days 00:00:21,1,9cc8a90a8f8b6a7b56a486b612b40efa,c3cfdc648177fdbbbb35635a37472c53,2018-08-08 14:55:21,129.99,18.71,1,voucher,1,148.70,29438818bd857d086f0854aafd249ac5,5.0,NaN,NaN,2018-08-10 00:00:00,2018-08-13 12:00:17
1173,8d3cd96a636e59a3a47b52cedd6e9bbd,433d2182c7ede2cd82ab34f10ba26e2e,delivered,2018-06-20 14:48:16,2018-06-20 15:19:11,2018-06-20 15:18:00,2018-06-26 23:51:00,2018-07-18,0 days 00:01:11,1,0e9083f06ff39f2cfe1ba2c8f589f591,59b22a78efb79a4797979612b885db36,2018-06-26 15:19:11,239.00,25.83,1,credit_card,5,264.83,c5cce3fe2dfb9c36b84a5267af6ca6e8,5.0,NaN,NaN,2018-06-27 00:00:00,2018-06-30 09:01:06
625,b1790ff077b7630f790bba65c012c64b,3f2d49651aac6322e00f50492dab09dc,delivered,2018-06-20 15:04:55,2018-06-20 15:40:12,2018-06-20 15:39:00,2018-06-25 15:52:39,2018-07-24,0 days 00:01:12,1,17878434ca0082537806de545c2f0351,0b90b6df587eb83608a64ea8b390cf07,2018-06-28 15:31:45,140.90,14.33,1,credit_card,1,155.23,cbf8153f1e582ddeb5ab910796de8852,5.0,NaN,NaN,2018-06-26 00:00:00,2018-06-26 19:21:37
480,6b3c0661b32905af6722a962d2fac779,4f640c97c9198455b707c9d8a99f334f,delivered,2018-07-27 13:30:05,2018-07-27 13:45:13,2018-07-27 13:44:00,2018-08-01 11:52:46,2018-08-17,0 days 00:01:13,1,6f6d0c21d57a3bac0a6dc84cae4c0bcb,d98eec89afa3380e14463da2aabaea72,2018-07-31 13:45:13,79.99,16.75,1,credit_card,1,96.74,4e300cd11efe6bcbca600eebe32c8597,5.0,NaN,NaN,2018-08-02 00:00:00,2018-08-05 22:21:17
630,9256fda5584ec43f6212f63bdf17b2ec,2b3c93b880505c1bbc5b3de1dd0ace3d,delivered,2018-06-27 11:34:34,2018-06-27 11:57:21,2018-06-27 11:56:00,2018-06-28 18:26:27,2018-07-16,0 days 00:01:21,1,36c554eb2e204a1db6000ed146c9fbde,73b8eb4a9a729d4019b24ed1be748cbf,2018-07-05 11:57:21,116.25,8.07,1,credit_card,2,124.32,af798205615e03a298174ac9cacb9bd9,5.0,Muito bom,Muito bom,2018-06-29 00:00:00,2018-07-03 11:00:08
661,b7d794a0d512566c47d3ffb585bb3805,970922c6488462df7d8b02576a219263,delivered,2018-08-15 12:20:55,2018-08-15 12:35:25,2018-08-15 12:34:00,2018-08-18 01:44:44,2018-08-28,0 days 00:01:25,1,4608a449f45011c89095f36265e75ec3,dc7192adf8ba09569261f4a8d576afe0,2018-08-17 12:35:25,98.00,13.35,1,credit_card,3,111.35,a21649ce25ab00bea1d24167f6fa0ef0,5.0,ÓTIMA,PRODUTO OK E CHEGOU EM UM ÓTIMO PRAZO,2018-08-18 00:00:00,2018-08-20 11:55:14
889,404d71e18721243380c994069a7f411a,a1e91b297d20335ee3a05686ce77cd75,delivered,2018-07-24 13:06:23,2018-07-24 13:44:29,2018-07-24 13:43:00,2018-07-26 20:41:13,2018-08-07,0 days 00:01:29,1,730e0343a68b7cebc35ad4d14cf9f7af,31561f325664a8a7aba4c8d0c3a9b3db,2018-07-30 13:32:03,54.90,15.48,1,credit_card,1,70.38,53812a7b3fe0c7db2eb15997d71ca144,4.0,NaN,NaN,2018-07-27 00:00:00,2018-07-30 18:40:22
343,7dda251cd0ccf3b0e0ff56d62b779fdc,d506c0d4abd2528e9fddb923894a9dda,delivered,2017-05-21 08:00:14,2017-05-22 10:41:43,2017-05-22 10:40:10,2017-05-23 08:49:42,2017-06-02,0 days 00:01:33,1,c73628b1c3144b2e0c9c88072f21a213,3f995f07c49d0d55a99d5c54957f7d81,2017-05-25 09:32:55,99.00,7.95,1,credit_card,2,106.95,32ffc1fd1c264589dcd66dac07113649,5.0,NaN,tudo dez\r\n,2017-05-24 00:00:00,2017-05-24 19:16:43
454,99edc58dc1d02b11403a10338c3a696c,c4784d77ab20f22ff6a8cf8140d43b42,delivered,2018-05-15 14:21:56,2018-05-15 14:35:58,2018-05-15 14:34:00,2018-05-18 16:12:14,2018-05-29,0 days 00:01:58,1,804b8b7fa3fa43cfed7ceaa5fa0f6898,8a432f4e5b471f8da497d7dc517666e2,2018-05-17 14:35:58,64.00,12.89,1,credit_card,3,76.89,0986b6917a5657ef5c4f956788d53fcf,5.0,super

In [38]:
result.sort_values(
    'review_score',
    ascending=True
).head(30)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_delay,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
987,44f9bb85e9dfb3f106dd017fbd8b8006,213133b1f3d6d32a0380333e4ff1102c,delivered,2018-04-20 22:20:00,2018-04-24 18:10:37,2018-04-23 20:08:23,2018-05-01 16:02:34,2018-06-01,0 days 22:02:14,1,cb81df0e3ccece253557f2a07df4727e,669ae81880e08f269a64487cfb287169,2018-04-25 23:31:20,45.00,22.06,1,credit_card,3,134.12,244bd9c73d2cf2edb0b9de23fd86e58d,1.0,Entrega incompleta,"Paguei dois frascos, na nota constam dois e de...",2018-05-02 00:00:00,2018-05-02 20:10:47
640,91be51c856a90d7efe86cf9d082d6ae3,637321037fb8b34323ce7fd8aab4a0f1,delivered,2018-06-12 15:58:03,2018-06-12 16:32:26,2018-06-12 15:44:00,2018-06-15 15:42:07,2018-07-12,0 days 00:48:26,3,b7351032045cba592a6d97b36f5fd8e3,ce69a8021d18961dd2a40269b7c2c293,2018-06-18 16:30:57,99.99,19.16,1,credit_card,18,397.90,84775964cfaf2e4fa80f8657d4cdede8,1.0,Problemas,Entregaram apenas um produto dos quatro itens ...,2018-06-16 00:00:00,2018-06-16 19:06:19
641,91be51c856a90d7efe86cf9d082d6ae3,637321037fb8b34323ce7fd8aab4a0f1,delivered,2018-06-12 15:58:03,2018-06-12 16:32:26,2018-06-12 15:44:00,2018-06-15 15:42:07,2018-07-12,0 days 00:48:26,4,ae3e0cf8b9e4a3a027fc5d4b0a3eb2a0,596849622429351f47b32e6cae1055ff,2018-06-18 16:30:57,39.90,0.96,1,credit_card,18,397.90,84775964cfaf2e4fa80f8657d4cdede8,1.0,Problemas,Entregaram apenas um produto dos quatro itens ...,2018-06-16 00:00:00,2018-06-16 19:06:19
1414,7889e9a569a5488a0c9d93381a13d5ba,d49398157dec1751afd66073d61c5443,delivered,2018-07-04 15:19:13,2018-07-05 16:13:18,2018-07-05 13:59:00,2018-07-10 20:28:46,2018-08-01,0 days 02:14:18,1,5c3aca5c078a343b3b873362e7ac2fcc,056b4ada5bbc2c50cc7842547dda6b51,2018-07-10 15:31:13,175.99,20.42,1,credit_card,8,196.41,97414d1c7068513bbafd5e68a0e4145e,1.0,MEU PRODUTO NAO CHEGOU,Gostaria de uma posição porque meu produto nao...,2018-07-11 00:00:00,2018-07-13 11:11:55
883,e9ce8d3f379bf3d70e4af85174e1841f,383e0c402beac204dc7ad8633cc49ae8,shipped,2017-07-29 14:44:04,2017-08-02 08:34:47,2017-08-01 18:12:05,NaT,2017-08-24,0 days 14:22:42,1,d66815037fba03ebe1e3bcb9967723aa,7aa4334be125fcdd2ba64b3180029f14,2017-08-07 04:42:59,120.99,18.10,1,boleto,1,139.09,173434db869380f2ddd9af8e250a388e,1.0,NaN,NaN,2017-08-27 00:00:00,2017-08-28 18:11:57
1193,be555543375e799bd0f439ffc16134ab,ec17a9a3e1a147308f1e70ab736e787b,delivered,2018-07-03 10:11:40,2018-07-05 16:06:58,2018-07-04 14:44:00,2018-07-05 19:06:43,2018-07-23,1 days 01:22:58,2,bdcf6a834e8faa30dac3886c7a58e92e,2a84855fd20af891be03bc5924d2b453,2018-07-06 08:32:13,35.90,15.32,1,boleto,1,102.44,fdd3fea45535b3b6e2a9763bdf22fcd6,1.0,Ruim,Comprei duas balanças e so entregaram uma!!,2018-07-06 00:00:00,2018-07-06 22:30:11
1192,be555543375e799bd0f439ffc16134ab,ec17a9a3e1a147308f1e70ab736e787b,delivered,2018-07-03 10:11:40,2018-07-05 16:06:58,2018-07-04 14:44:00,2018-07-05 19:06:43,2018-07-23,1 days 01:22:58,1,bdcf6a834e8faa30dac3886c7a58e92e,2a84855fd20af891be03bc5924d2b453,2018-07-06 08:32:13,35.90,15.32,1,boleto,1,102.44,fdd3fea45535b3b6e2a9763bdf22fcd6,1.0,Ruim,Comprei duas balanças e so entregaram uma!!,2018-07-06 00:00:00,2018-07-06 22:30:11
290,b5ee9c8a7edd714067ee2db82c6da589,3cb367d10216cbaec82d09ff0ceefeba,delivered,2018-04-23 12:38:56,2018-04-24 17:28:58,2018-04-24 17:13:05,2018-04-30 21:34:25,2018-05-22,0 days 00:15:53,2,086351823300e0339f6955b27998c186,33a6f4b1e7cdc205511e76ba1b6e0186,2018-04-30 13:30:59,115.00,39.89,1,credit_card,6,268.13,1a5c7eff049b52c66f30b23f8d5d85eb,1.0,Falha na entrega,Comprei dois produtos e só entregou um.,2018-05-01 00:00:00,2018-05-02 01:32:21
639,91be51c856a90d7efe86cf9d082d6ae3,637321037fb8b34323ce7fd8aab4a0

# Overall Assessment

The available data is suitable for further business analysis.

The investigation identified several expected missing values and a small number of timestamp inconsistencies. Manual validation indicates that most anomalies are caused by incomplete system recording rather than operational failures.

## Key Findings

- Missing review comments are expected because not every customer leaves written feedback.
- Product attribute fields contain limited missing values that can be handled during preparation.
- Only eight delivered orders are missing customer delivery timestamps, with customer reviews confirming successful delivery.
- No unexpected duplicate records were identified in transactional datasets.
- Timestamp columns require conversion before time-based analysis.
- Most chronological relationships are internally consistent.
- A limited number of timestamp anomalies were identified, but they represent isolated edge cases rather than systemic issues.

## Recommendation

Proceed with data preparation and feature engineering. The dataset is sufficiently complete and reliable to support customer behaviour, delivery performance, revenue, seller performance, and customer satisfaction analysis.